# 04 Part I summary: does a generative model beat duplicating the rare class?
Recomputes a subset of the Part I headline numbers from the result tables of `part1_results.tgz`. These tables are per-run outputs of the augmentation experiments (training the generators takes days, so they are not rerun here). The code that turns them into the numbers below is in `part1/`.

Covered:
1. 88-dataset panel: 51 of 88 datasets with a significant win of a published generator or our scDesign3 ablation at the default threshold, datasets where scDesign3 without copula (our ablation) beats duplication (family-wise max-T), no deep generator wins, the chance ceiling, deep-generator mean change and Benjamini-Hochberg losses, duplication against no augmentation.
2. Duplication equals sample weighting: 160 (seed, type) units.
3. 31-type AIDA run: random-forest jitter gain, logistic-regression duplication against no augmentation. These values are not in the paper, and the paper's 31-type statement (matched-operating-point paragraph) is not recomputed.

Not covered (inputs not in the release or not recomputed here): the AIDA, HLCA and lupus runs; unmodified scDesign3, noise and SMOTE augmentation; the per-test counts of Fig. 1a; pooled precision in the winning datasets; the CFGen decoder result; the gene-selection numbers (HVGs on all cells against training cells only); foundation-model embeddings; the CFGen and scDiffusion reproductions; the false-positive and one-vs-rest analyses; the 31-type duplication against no augmentation values of the paper.


In [1]:
import os, sys, warnings

warnings.filterwarnings("ignore")
REPO = os.path.abspath(os.environ.get("REPO", ".."))
ASSETS = os.path.abspath(os.environ.get("ASSETS", os.path.join(REPO, "assets")))
sys.path.insert(0, REPO)
sys.dont_write_bytecode = True
import json, numpy as np, pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
EXP = f"{REPO}/eval/expected"
from part1 import panel, probe31

P1 = f"{ASSETS}/part1_results"

## 1. The 88-dataset panel
Per dataset, d = recall at the matched operating point (rec200) of an augmentation minus duplication, per probe type. P2 is a one-sided studentized max-T sign-flip over every published generator and classifier (256 exact flips, family-wise error 0.05). About two minutes.

In [2]:
T, P2, Ac = panel.run(
    f"{P1}/panel_runs", f"{P1}/panel_eligible.csv", metrics=("rec200", "rec_argmax")
)
Ac = Ac.set_index(["metric", "clf", "method"])
p2 = P2[P2.metric == "rec200"]
win = p2[p2.p_maxT < 0.05]
print(
    f"datasets: {p2.ds.nunique()}, tissues: {T.tissue.nunique()}; scDesign3 without copula (our ablation) beats duplication in {len(win)} datasets (chance ceiling {0.05 * len(p2):.1f})"
)
print(
    win[["ds", "tissue", "method", "clf", "d", "pos", "p_maxT"]]
    .round(4)
    .to_string(index=False)
)
assert (
    p2.ds.nunique() == 88
    and T.tissue.nunique() == 31
    and len(win) == 4
    and set(win.method) == {"nb"}
    and round(0.05 * len(p2), 1) == 4.4
)

datasets: 88, tissues: 31; scDesign3 without copula (our ablation) beats duplication in 4 datasets (chance ceiling 4.4)
                                  ds      tissue method clf      d  pos  p_maxT
15d374d6-0dfd-4d8e-ade7-81f73dc921ee musculature     nb mlp 0.0431    7  0.0469
ac0c6561-7a48-4185-af6f-af799f699172       brain     nb  rf 0.0817    8  0.0430
b252b015-b488-4d5c-b16e-968c13e48a2c     abdomen     nb mlp 0.0119    8  0.0078
f1606894-59df-4794-a37f-baa7c6fb6de1       heart     nb  rf 0.0896    8  0.0273


In [3]:
A_ = panel.run(f"{P1}/panel_runs", f"{P1}/panel_eligible.csv", metrics=("rec_argmax",))[
    1
]
n51 = int((A_.p_maxT < 0.05).sum())
assert A_.ds.nunique() == 88
print(
    f"default threshold: {n51} of 88 datasets show a significant win of a published generator or our scDesign3 ablation"
)
assert n51 == 51

default threshold: 51 of 88 datasets show a significant win of a published generator or our scDesign3 ablation


In [4]:
a = Ac.reset_index()
a = a[(a.metric == "rec200") & (a.method.isin(panel.DEEP))]
print(
    f"deep generators: mean change {a.mean_d.max():+.3f} to {a.mean_d.min():+.3f}; BH losses on {a.ds_loss.min()} to {a.ds_loss.max()} of 88 datasets; BH wins {a.ds_win.sum()}"
)
assert (
    (round(a.mean_d.max(), 3), round(a.mean_d.min(), 3)) == (-0.045, -0.301)
    and (a.ds_loss.min(), a.ds_loss.max()) == (23, 87)
    and a.ds_win.sum() == 0
)

deep generators: mean change -0.045 to -0.301; BH losses on 23 to 87 of 88 datasets; BH wins 0


Duplication minus no augmentation (the sign of the "real" row flipped), at the matched operating point and at the default threshold. BH over the eight rows.

In [5]:
p3 = Ac.reset_index()
p3 = (
    p3[(p3.method == "real") & (p3["sub"] == "all")]
    .set_index(["metric", "clf"])
    .sort_index()
)
p3["dup_minus_noaug"] = -p3.mean_d
p3[["ds_loss", "ds_win"]] = p3[["ds_win", "ds_loss"]].values
print(p3[["dup_minus_noaug", "ds_loss", "ds_win", "p_ds"]].round(3))
exp = {
    ("rec200", "knn"): -0.025,
    ("rec200", "lr"): 0.007,
    ("rec200", "mlp"): 0.010,
    ("rec200", "rf"): -0.014,
    ("rec_argmax", "knn"): 0.288,
    ("rec_argmax", "lr"): 0.066,
    ("rec_argmax", "rf"): 0.051,
    ("rec_argmax", "mlp"): 0.041,
}
assert all(round(p3.loc[k, "dup_minus_noaug"], 3) == v for k, v in exp.items())

                dup_minus_noaug  ds_loss  ds_win  p_ds
metric     clf                                        
rec200     knn           -0.025       16       0   0.0
           lr             0.007        2      12   0.0
           mlp            0.010        1      23   0.0
           rf            -0.014        2       0   0.0
rec_argmax knn            0.288        0      85   0.0
           lr             0.066        0      81   0.0
           mlp            0.041        0      60   0.0
           rf             0.051        0      68   0.0


## 2. Duplication is sample weighting
Fitting a logistic regression on the duplicated rows or on the original rows with sample weights equal to the copy counts gives the same classifier: 20 seeds x 8 probe types.

In [6]:
d = pd.read_csv(f"{P1}/dup_weight.csv")
g = d.groupby("seed").first()
print(
    len(d),
    "units; mean recall difference %+.5f; largest %.3f; smallest argmax agreement %.4f"
    % (
        (d.rec_dup - d.rec_w).mean(),
        (d.rec_dup - d.rec_w).abs().max(),
        g.argmax_agree.min(),
    ),
)
assert (
    len(d) == 160
    and round((d.rec_dup - d.rec_w).mean(), 5) == 0.00006
    and g.argmax_agree.min() >= 0.993
)

160 units; mean recall difference +0.00006; largest 0.025; smallest argmax agreement 0.9932


## 3. The 31-type AIDA run (values not printed in the paper)
Unit = cell type, two-sided sign-flip over 31 types, BH over the 20 primary tests (5 augmentations x 4 classifiers).

In [7]:
R = probe31.table(f"{P1}/probe31_runs.csv")
a = R[R.part == "all"].set_index(["clf", "method"])
j = a.loc[("rf", "jitter_s0.5")]
lr = a.loc[("lr", "real")]
print(
    f"RF with Gaussian jitter minus duplication: {j.d:+.3f} ({j.types_pos}/31 types); LR no augmentation minus duplication: {lr.d:+.4f} (q = {lr.q_primary:.3f}, {lr.types_pos}/31 types favour no augmentation)"
)
assert (
    round(j.d, 3) == 0.041
    and j.types_pos == 30
    and round(lr.d, 4) == 0.0055
    and round(lr.q_primary, 3) == 0.027
)

RF with Gaussian jitter minus duplication: +0.041 (30/31 types); LR no augmentation minus duplication: +0.0055 (q = 0.027, 22/31 types favour no augmentation)


For logistic regression, duplication is behind no augmentation by 0.0055.